In [1]:
#Config 1: Minimal - Single hidden layer
config_1 = {
    'layers': [4],
    'activation': ['relu'],
    'dropout': [0.1]
}

# Config 2: Small but effective
config_2 = {
    'layers': [8, 4], 
    'activations': ['relu', 'relu'],
    'dropout': [0.2, 0.1]
}

# Config 3: Slightly wider
config_3 = {
    'layers': [16, 8], 
    'activations': ['tanh', 'relu'],
    'dropout': [0.3, 0.2]
}

In [2]:
# Config 4: Three layers, moderate size
config_4 = {
    'layers': [16, 8, 4], 
    'activations': ['relu', 'relu', 'relu'],
    'dropout': [0.2, 0.2, 0.1]
}

# Config 5: Uniform layer size
config_5 = {
    'layers': [12, 12, 12], 
    'activations': ['tanh', 'tanh', 'relu'],
    'dropout': [0.2, 0.2, 0.2]
}

# Config 6: Mixed activations
config_6 = {
    'layers': [32, 16, 8], 
    'activations': ['sigmoid', 'relu', 'tanh'],
    'dropout': [0.3, 0.2, 0.1]
}


In [3]:
# Config 7: Deep narrow network
config_7 = {
    'layers': [8, 8, 8, 8, 4], 
    'activations': ['relu', 'relu', 'relu', 'relu', 'relu'],
    'dropout': [0.1, 0.1, 0.1, 0.1, 0.1]
}

# Config 8: Wide then narrow
config_8 = {
    'layers': [64, 32, 16, 4], 
    'activations': ['relu', 'relu', 'tanh', 'relu'],
    'dropout': [0.4, 0.3, 0.2, 0.1]
}


## Implementation Framework

In [4]:
import tensorflow as tf
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score, accuracy_score


def create_nn_model(config, input_dim=5):
    """Create NN model based on configuration"""
    model = tf.keras.Sequential()
    
    # Input layer
    model.add(tf.keras.layers.Dense(
        config['layers'][0], 
        activation=config['activations'][0],
        input_shape=(input_dim,)
    ))
    
    # Add dropout if specified
    if 'dropout' in config and len(config['dropout']) > 0:
        model.add(tf.keras.layers.Dropout(config['dropout'][0]))
    
    # Hidden layers
    for i in range(1, len(config['layers'])):
        model.add(tf.keras.layers.Dense(
            config['layers'][i], 
            activation=config['activations'][i]
        ))
        
        if 'dropout' in config and i < len(config['dropout']):
            model.add(tf.keras.layers.Dropout(config['dropout'][i]))
    
    # Output layer (always sigmoid for binary classification)
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    
    return model

def evaluate_nn_config(config, X_train, y_train, X_test, y_test, config_name, model_container):
    """Evaluate a single NN configuration"""
    model = create_nn_model(config)
    model_container[config_name] = model
    # Compile model
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )
    
    # Train with early stopping
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )
    
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Evaluate
    y_pred_proba = model.predict(X_test).flatten()
    
    # Calculate metrics
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    pr_auc = auc(recall, precision)
    
    # Get final metrics from training
    final_val_loss = min(history.history['val_loss'])
    epochs_trained = len(history.history['loss'])

    Y_actual = y_test
    Y_prob = y_pred_proba

     # --- Compute best F1 score by sweeping thresholds ---
    thresholds = np.linspace(0, 1, 101)  # thresholds from 0.0 to 1.0
    f1_scores = [f1_score(Y_actual, (Y_prob >= t).astype(int)) for t in thresholds]
    best_f1 = max(f1_scores)
    best_threshold = thresholds[np.argmax(f1_scores)]
    
    # Binary predictions using the best threshold
    Y_pred_best = (Y_prob >= best_threshold).astype(int)
    # --- ROC curve ---
    fpr, tpr, roc_thresholds = roc_curve(Y_actual, Y_prob)
    

    # --- Precision-Recall curve ---
    precision, recall, pr_thresholds = precision_recall_curve(Y_actual, Y_prob)
    avg_precision = average_precision_score(Y_actual, Y_prob)
    acc = accuracy_score(Y_actual, Y_pred_best)

    results = {
        'config_name': config_name,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'val_loss': final_val_loss,
        'epochs': epochs_trained,
        'model': model,
        'history': history,
        "F1": best_f1,
        "Best_Threshold": best_threshold,
        "AUC": roc_auc_score(Y_actual, Y_prob),
        "Average_Precision": avg_precision,
        "Accuracy": acc

    }
    
    return results


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
#import xgboost as xgb
from sklearn.metrics import precision_recall_curve, auc
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import lightgbm as lgb
import os


df = pd.read_csv("..\..\Meta Model Dataset\Training_For_Meta_Model.csv")
# Define target variable (presence of cardiovascular disease)
y = df["Cardiovascular Disease"]

# Select relevant features
features = ["Age", "Systolic Blood Pressure", "Diastolic Blood Pressure", "Cholesterol Level", "Glucose Level", "Smoking Status", "Alcohol Intake", "Physical Activity"]
X = df[features]

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)
lifestyle_columns_to_drop = ['Cholesterol Level', 'Diastolic Blood Pressure', 'Systolic Blood Pressure', 'Glucose Level', 'Age', 'Gender', 'BMI', 'id']
health_columns_to_drop = ['Smoking Status', 'Physical Activity', 'Alcohol Intake', 'Age', 'Gender', 'BMI', 'id']

X = df.drop(['Cardiovascular Disease'], axis=1)
Y = df['Cardiovascular Disease']


X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2, 
    random_state=42,
    stratify=Y 
)


health_dataset = X_train.drop(columns=health_columns_to_drop)
lifestyle_dataset = X_train.drop(columns=lifestyle_columns_to_drop)


# Load the trained models
#model_A = lgb.Booster(model_file="saved_models_tausif/lightgbm_model_A.txt")
# model_B = lgb.Booster(model_file="Models B/lightgbm_model.txt")
# model_A = lgb.Booster(model_file="Saved Models A/lightgbm_model.txt")
# Egula RETRAINED models 
model_A = lgb.Booster(model_file="lightgbm_model_A.txt")
model_B = lgb.Booster(model_file="lightgbm_model_B.txt")

predictions_A = model_A.predict(lifestyle_dataset)
predictions_B = model_B.predict(health_dataset)


print(predictions_A)
print(predictions_B)

[0.36195349 0.36195349 0.30475363 ... 0.30475363 0.30475363 0.30475363]
[0.22774369 0.9214363  0.8089465  ... 0.24437861 0.58089957 0.24437861]


<>:14: SyntaxWarning: invalid escape sequence '\.'
<>:14: SyntaxWarning: invalid escape sequence '\.'
C:\Users\adibs\AppData\Local\Temp\ipykernel_6868\2945996712.py:14: SyntaxWarning: invalid escape sequence '\.'
  df = pd.read_csv("..\..\Meta Model Dataset\Training_For_Meta_Model.csv")


In [7]:
import numpy as np
# Prepare meta-learner data (probabilities from Model A and B)
a_g_BMI = X_train[['Age', 'Gender', 'BMI']].values
meta_X = np.column_stack([predictions_A, predictions_B, a_g_BMI])

meta_y = Y_train.values

# Split for meta-learner training
X_train_meta, X_test_meta, y_train_meta, y_test_meta = train_test_split(
    meta_X, meta_y, test_size=0.2, random_state=42, stratify=meta_y
)

# Test all configurations
configurations = {
    'Simple_4': {'layers': [4], 'activations': ['relu'], 'dropout': [0.1]},
    'Small_8-4': {'layers': [8, 4], 'activations': ['relu', 'relu'], 'dropout': [0.2, 0.1]},
    'Medium_16-8': {'layers': [16, 8], 'activations': ['tanh', 'relu'], 'dropout': [0.3, 0.2]},
    'Triple_16-8-4': {'layers': [16, 8, 4], 'activations': ['relu', 'relu', 'relu'], 'dropout': [0.2, 0.2, 0.1]},
    'Uniform_12x3': {'layers': [12, 12, 12], 'activations': ['tanh', 'tanh', 'relu'], 'dropout': [0.2, 0.2, 0.2]},
    'Mixed_32-16-8': {'layers': [32, 16, 8], 'activations': ['sigmoid', 'relu', 'tanh'], 'dropout': [0.3, 0.2, 0.1]},
    'Deep_8x5': {'layers': [8, 8, 8, 8, 4], 'activations': ['relu']*5, 'dropout': [0.1]*5},
    'Wide_64-32-16-4': {'layers': [64, 32, 16, 4], 'activations': ['relu', 'relu', 'tanh', 'relu'], 'dropout': [0.4, 0.3, 0.2, 0.1]}
}

results = {}
model_container = {}
for config_name, config in configurations.items():
    print(f"Testing {config_name}...")
    result = evaluate_nn_config(
        config, X_train_meta, y_train_meta, X_test_meta, y_test_meta, config_name, model_container
    )
    results[config_name] = result


Testing Simple_4...


C:\Users\adibs\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
Testing Small_8-4...


C:\Users\adibs\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
Testing Medium_16-8...


...

69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Testing Deep_8x5...


C:\Users\adibs\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Testing Wide_64-32-16-4...


C:\Users\adibs\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [8]:
import pandas as pd

# Create results comparison
comparison_data = []
for name, result in results.items():
    comparison_data.append({
        'Configuration': name,
        'ROC-AUC': result['roc_auc'],
        'PR-AUC': result['pr_auc'],
        'Val_Loss': result['val_loss'],
        'Epochs': result['epochs'],
        'F1': result['F1'],
        'Best_Threshold': result['Best_Threshold'],
        'AUC': result['AUC'],
        'Average_Precision': result['Average_Precision'],
        'Accuracy': result['Accuracy']
    })

results_df = pd.DataFrame(comparison_data)
results_df = results_df.sort_values('PR-AUC', ascending=False)
print(results_df)


     Configuration   ROC-AUC    PR-AUC  Val_Loss  Epochs        F1  \
5    Mixed_32-16-8  0.790637  0.774404  0.551229      61  0.730689   
0         Simple_4  0.783990  0.764402  0.561963     100  0.728897   
6         Deep_8x5  0.780412  0.759789  0.586504      58  0.730891   
3    Triple_16-8-4  0.771270  0.753192  0.570878      76  0.716948   
7  Wide_64-32-16-4  0.500000  0.748055  0.693686      33  0.663200   
1        Small_8-4  0.513247  0.519294  0.691496      17  0.663809   
2      Medium_16-8  0.486929  0.487580  0.693076      18  0.663200   
4     Uniform_12x3  0.483637  0.481670  0.692889      18  0.663200   

   Best_Threshold       AUC  Average_Precision  Accuracy  
5            0.34  0.790637           0.774707  0.704805  
0            0.41  0.783990           0.764860  0.722197  
6            0.35  0.780412           0.755811  0.727689  
3            0.39  0.771270           0.753785  0.708009  
7            0.00  0.500000           0.496110  0.496110  
1            0.

In [9]:
best_config_name = "Triple_16-8-4"
best_config = configurations[best_config_name]
best_model = create_nn_model(best_config)

results = {}
model_container_best = {}

print(f"Testing {best_config_name}...")
result = evaluate_nn_config(
    best_config, X_train_meta, y_train_meta, X_test_meta, y_test_meta, best_config_name, model_container_best
)
results[best_config_name] = result  # <-- Use string key here
# ...existing code...
comparison_data = []
for name, result in results.items():
    comparison_data.append({
        'Configuration': name,
        'ROC-AUC': result['roc_auc'],
        'PR-AUC': result['pr_auc'],
        'Val_Loss': result['val_loss'],
        'Epochs': result['epochs'],
        'F1': result['F1'],
        'Best_Threshold': result['Best_Threshold'],
        'AUC': result['AUC'],
        'Average_Precision': result['Average_Precision'],
        'Accuracy': result['Accuracy']
    })

results_df = pd.DataFrame(comparison_data)
results_df = results_df.sort_values('PR-AUC', ascending=False)
print(results_df)



C:\Users\adibs\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Testing Triple_16-8-4...
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
   Configuration   ROC-AUC    PR-AUC  Val_Loss  Epochs        F1  \
0  Triple_16-8-4  0.776997  0.761213  0.574734      58  0.724295   

   Best_Threshold       AUC  Average_Precision  Accuracy  
0            0.43  0.776997           0.760879  0.722654  


In [10]:
model_container['Simple_4'].compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)
#evaluate
model_container['Simple_4'].evaluate(X_test_meta, y_test_meta)
#save the model
model_container['Simple_4'].save('simple_4_model.h5')

69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7254 - loss: 0.5648 - precision: 0.7516 - recall: 0.6670  
